### ETL in Databricks
This notebook gives some basic commands for performing necessary ETL functions. Use this to create a medallion ETL pipeline using the financial dataset.

You should have:
 - bronze schema with tables for each set
 - silver schema with cleaned and formatted tables
 - gold schema with aggregated tables (to answer the questions in the notion page)

The notebook will be used as the source a daily job to refresh the pipeline (The whole notebook will be executed) and a dashboard will be created using the gold tables as source data.


In [0]:
%sql
-- Use the default catalog (main)
USE CATALOG jarvis_server_1;

-- Create schemas for medallion architecture
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

-- Verify schemas
SHOW SCHEMAS;


databaseName
bronze
default
gold
information_schema
silver


In [0]:
url = "jdbc:sqlserver://jarvis-server-1.database.windows.net:1433;database=Financial-Database;encrypt=true;trustServerCertificate=false;loginTimeout=30;"
user = "jarvis-server-1"
password = "Lecture6"

# Connect via Spark
df_tables = spark.read.format("jdbc") \
    .option("url", url) \
    .option("dbtable", "INFORMATION_SCHEMA.TABLES") \
    .option("user", user) \
    .option("password", password) \
    .load()

display(df_tables)


---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-5999673723621170>, line 13
      5 # Connect via Spark
      6 df_tables = spark.read.format("jdbc") \
      7     .option("url", url) \
      8     .option("dbtable", "INFORMATION_SCHEMA.TABLES") \
      9     .option("user", user) \
     10     .option("password", password) \
     11     .load()
---> 13 display(df_tables)

File /databricks/python_shell/lib/dbruntime/display.py:135, in Display.display(self, input, *args, **kwargs)
    133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
--> 135     if input.isStreaming:
    136         handleStreamingConnectDataFramePy4j(input, self.entry_point, kwargs)
    137     else:

File /usr/lib/python3.12/functools.py:995, in cached_property.__get__(self, instance, owner)
    993 val = cache.get(self.attrname, _NOT_FOUND)
    994 if

In [0]:
from pyspark.sql.functions import from_json, explode, col

# -------------------------------
# Paths to volumes
# -------------------------------
base_path = "/Volumes/jarvis_server_1/default/volume"
mcc_path = f"{base_path}/mcc_codes.json"
labels_path = f"{base_path}/train_fraud_labels.json"

# -------------------------------
# 1. Read + normalize MCC codes into df_mcc
# -------------------------------
df_mcc = (
    spark.read.text(mcc_path)  # read as text because it's a JSON map
    .select(from_json(col("value"), "map<string,string>").alias("mcc_map"))
    .select(explode(col("mcc_map")).alias("mcc_code", "mcc_description"))
    .withColumn("mcc_code", col("mcc_code").cast("int"))
)

# -------------------------------
# 2. Read fraud labels into df_labels
# -------------------------------
df_labels = spark.read.json(labels_path)  # standard row-based JSON

# -------------------------------
# Now both volumes are stored in variables
# -------------------------------
print("df_mcc and df_labels are ready!")
display(df_mcc)
display(df_labels)


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# =========================================
# BRONZE INGESTION 
# =========================================

from pyspark.sql.functions import explode, col, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, MapType
)

VOLUME_PATH = "/Volumes/jarvis_server_1/default/volume"

# -----------------------------------------
# 1. MCC CODES (root-level map JSON)
# -----------------------------------------
mcc_schema = StructType([
    StructField(
        "data",
        MapType(StringType(), StringType()),
        True
    )
])

df_mcc_raw = spark.read \
    .schema(mcc_schema) \
    .json(f"{VOLUME_PATH}/mcc_codes.json")

df_mcc = df_mcc_raw \
    .select(explode(col("data")).alias("mcc", "description")) \
    .withColumn("_ingest_ts", current_timestamp())

df_mcc.write \
    .mode("overwrite") \
    .saveAsTable("jarvis_server_1.bronze.mcc_codes")

# -----------------------------------------
# 2. FRAUD LABELS (nested map JSON)
# -----------------------------------------
labels_schema = StructType([
    StructField(
        "target",
        MapType(StringType(), StringType()),
        True
    )
])

df_labels_raw = spark.read \
    .schema(labels_schema) \
    .json(f"{VOLUME_PATH}/train_fraud_labels.json")

df_labels = df_labels_raw \
    .select(explode(col("target")).alias("transaction_id", "fraud_label")) \
    .withColumn(
        "is_fraud",
        (col("fraud_label") == "Yes").cast(IntegerType())
    ) \
    .drop("fraud_label") \
    .withColumn("_ingest_ts", current_timestamp())

df_labels.write \
    .mode("overwrite") \
    .saveAsTable("jarvis_server_1.bronze.train_fraud_labels")

# -----------------------------------------
# 3. WORKSPACE TABLES → BRONZE
# -----------------------------------------
df_cards = spark.table("jarvis_server_1.default.cards_data") \
    .withColumn("_ingest_ts", current_timestamp())

df_tx = spark.table("jarvis_server_1.default.transactions_data") \
    .withColumn("_ingest_ts", current_timestamp())

df_users = spark.table("jarvis_server_1.default.users_data") \
    .withColumn("_ingest_ts", current_timestamp())

df_cards.write.mode("overwrite").saveAsTable("jarvis_server_1.bronze.cards_data")
df_tx.write.mode("overwrite").saveAsTable("jarvis_server_1.bronze.transactions_data")
df_users.write.mode("overwrite").saveAsTable("jarvis_server_1.bronze.users_data")

# -----------------------------------------
# 4. VERIFY
# -----------------------------------------
spark.sql("SHOW TABLES IN jarvis_server_1.bronze").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
|  bronze|        cards_data|      false|
|  bronze|         mcc_codes|      false|
|  bronze|train_fraud_labels|      false|
|  bronze| transactions_data|      false|
|  bronze|        users_data|      false|
+--------+------------------+-----------+



In [0]:
# =========================================
# SILVER WORKFLOW – CLEAN + ENHANCE TRANSACTIONS
# =========================================

from pyspark.sql.functions import col, to_timestamp, trim, regexp_replace, current_timestamp, coalesce, lit
from pyspark.sql.types import DoubleType, IntegerType

# -----------------------------------------
# 1️⃣ Clean transactions_data
# -----------------------------------------
df_tx = spark.table("jarvis_server_1.bronze.transactions_data")

df_tx_silver = (
    df_tx
    .withColumn("transaction_ts", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("amount", regexp_replace(col("amount"), "[$,]", "").cast(DoubleType()))
    .withColumn("mcc", trim(col("mcc")))
    .filter(col("id").isNotNull())
    .dropDuplicates(["id"])
    .withColumn("_silver_ts", current_timestamp())
)

df_tx_silver.write.mode("overwrite").saveAsTable("jarvis_server_1.silver.transactions_data")


# -----------------------------------------
# 2️⃣ Clean cards_data
# -----------------------------------------
df_cards = spark.table("jarvis_server_1.bronze.cards_data")

df_cards_silver = (
    df_cards
    .withColumnRenamed("id", "card_id")
    .withColumn("card_brand", trim(col("card_brand")))
    .dropDuplicates(["card_id"])
    .withColumn("_silver_ts", current_timestamp())
)

df_cards_silver.write.mode("overwrite").saveAsTable("jarvis_server_1.silver.cards_data")


# -----------------------------------------
# 3️⃣ Clean users_data
# -----------------------------------------
df_users = spark.table("jarvis_server_1.bronze.users_data")

df_users_silver = (
    df_users
    .withColumnRenamed("id", "user_id")
    .dropDuplicates(["user_id"])
    .withColumn("_silver_ts", current_timestamp())
)

df_users_silver.write.mode("overwrite").saveAsTable("jarvis_server_1.silver.users_data")


# -----------------------------------------
# 4️⃣ Clean MCC codes
# -----------------------------------------
df_mcc = spark.table("jarvis_server_1.bronze.mcc_codes")

df_mcc_silver = (
    df_mcc
    .withColumn("mcc", trim(col("mcc")))
    .dropDuplicates(["mcc"])
    .withColumn("_silver_ts", current_timestamp())
)

df_mcc_silver.write.mode("overwrite").saveAsTable("jarvis_server_1.silver.mcc_codes")


# -----------------------------------------
# 5️⃣ Clean fraud labels
# -----------------------------------------
df_labels = spark.table("jarvis_server_1.bronze.train_fraud_labels")

df_labels_silver = (
    df_labels
    .withColumn("is_fraud", col("is_fraud").cast(IntegerType()))
    .dropDuplicates(["transaction_id"])
    .withColumn("_silver_ts", current_timestamp())
)

df_labels_silver.write.mode("overwrite").saveAsTable("jarvis_server_1.silver.train_fraud_labels")


# =========================================
# 6️⃣ Enhance transactions with MCC & fraud
# =========================================

df_tx_enriched = (
    df_tx_silver
    # join MCC description
    .join(df_mcc_silver.select("mcc", "description"), on="mcc", how="left")
    # join fraud labels
    .join(df_labels_silver.select("transaction_id", "is_fraud"), df_tx_silver.id == df_labels_silver.transaction_id, how="left")
    # fill null fraud with 0
    .withColumn("is_fraud", coalesce(col("is_fraud"), lit(0)).cast(IntegerType()))
)

# Save enhanced transactions table
df_tx_enriched.write.mode("overwrite").saveAsTable("jarvis_server_1.silver.transactions_enriched")


# -----------------------------------------
# 7️⃣ Verify Silver Tables
# -----------------------------------------
spark.sql("SHOW TABLES IN jarvis_server_1.silver").show()
display(df_tx_enriched)

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|  silver|          cards_data|      false|
|  silver|           mcc_codes|      false|
|  silver|  train_fraud_labels|      false|
|  silver|   transactions_data|      false|
|  silver|transactions_enri...|      false|
|  silver|          users_data|      false|
+--------+--------------------+-----------+



mcc,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,_ingest_ts,transaction_ts,_silver_ts,description,transaction_id,is_fraud
5300,13495843,2013-10-26T09:29:00.000Z,488,5413,123.21,Swipe Transaction,8044,Gregory,TX,78359.0,null,2026-02-17T14:04:06.549Z,2013-10-26T09:29:00.000Z,2026-02-17T14:39:39.017Z,null,null,0
4784,13496058,2013-10-26T10:09:00.000Z,859,2304,53.71,Online Transaction,39021,ONLINE,null,null,null,2026-02-17T14:04:06.549Z,2013-10-26T10:09:00.000Z,2026-02-17T14:39:39.017Z,null,13496058,0
5411,13496658,2013-10-26T11:55:00.000Z,1379,6068,16.02,Swipe Transaction,98372,Clinton Township,MI,48036.0,null,2026-02-17T14:04:06.549Z,2013-10-26T11:55:00.000Z,2026-02-17T14:39:39.017Z,null,13496658,0
4784,13497907,2013-10-26T15:54:00.000Z,1058,4102,31.58,Online Transaction,39021,ONLINE,null,null,null,2026-02-17T14:04:06.549Z,2013-10-26T15:54:00.000Z,2026-02-17T14:39:39.017Z,null,13497907,0
5813,13497919,2013-10-26T15:57:00.000Z,496,3428,22.03,Swipe Transaction,45344,Mesquite,TX,75149.0,null,2026-02-17T14:04:06.549Z,2013-10-26T15:57:00.000Z,2026-02-17T14:39:39.017Z,null,13497919,0
5499,13498506,2013-10-26T18:46:00.000Z,909,5232,66.0,Swipe Transaction,59935,Olympia,WA,98516.0,null,2026-02-17T14:04:06.549Z,2013-10-26T18:46:00.000Z,2026-02-17T14:39:39.017Z,null,13498506,0
5912,13498729,2013-10-26T20:21:00.000Z,1130,92,111.14,Swipe Transaction,81833,Federal Way,WA,98003.0,null,2026-02-17T14:04:06.549Z,2013-10-26T20:21:00.000Z,2026-02-17T14:39:39.017Z,null,13498729,0
5300,13498957,2013-10-26T21:55:00.000Z,1969,4981,83.22,Swipe Transaction,256,Bradenton,FL,34212.0,null,2026-02-17T14:04:06.549Z,2013-10-26T21:55:00.000Z,2026-02-17T14:39:39.017Z,null,13498957,0
4784,13499348,2013-10-27T04:38:00.000Z,1718,2029,29.14,Online Transaction,15143,ONLINE,null,null,null,2026-02-17T14:04:06.549Z,2013-10-27T04:38:00.000Z,2026-02-17T14:39:39.017Z,null,null,0
5812,13499562,2013-10-27T06:27:00.000Z,81,4525,1.58,Swipe Transaction,86809,Seguin,TX,78155.0,null,2026-02-17T14:04:06.549Z,2013-10-27T06:27:00.000Z,2026-02-17T14:39:39.017Z,null,13499562,0


In [0]:
df_silver.write.mode("overwrite").saveAsTable("catalog.schema.silver_table")

In [0]:
# Transform your silver dataframe to gold (aggregated, joined, etc.)
silver_df = spark.read.table("catalog.schema.silver_table")

gold_df = (silver_df
           .groupBy("address") # example
           .agg(sum("total_debt").alias("total_debt"))
           .orderBy(desc("total_debt"))
)

display(gold_df)

In [0]:
gold_df.write.mode("overwrite").saveAsTable("catalog.schema.gold_table")